[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IshtVibhu/iv-Mesa-ABM-Tutorial/blob/main/NB_05_SciPy.ipynb)

# Python for Computational Economics
## Notebook 05 — SciPy: Optimisation, Integration, and ODEs

**Prerequisites:** NB_01–NB_04

**What you will learn:**
- `scipy.optimize` — utility maximisation, profit maximisation, equilibrium finding
- `scipy.integrate` — consumer surplus, present value of continuous income streams
- `scipy.integrate.odeint` — Solow growth model, SIR epidemic model
- `scipy.stats` — hypothesis testing with economic data

In [ ]:
try:
    import scipy
    import numpy as np
    import matplotlib.pyplot as plt
except ImportError:
    !pip install scipy numpy matplotlib --quiet
    import scipy
    import numpy as np
    import matplotlib.pyplot as plt

from scipy import optimize, integrate, stats
plt.style.use("seaborn-v0_8-whitegrid")
print(f"SciPy {scipy.__version__}")

---
## 1. Utility Maximisation — Consumer Theory

A consumer maximises Cobb-Douglas utility `U(x1, x2) = x1^a * x2^(1-a)` subject to budget constraint `p1*x1 + p2*x2 = I`.

SciPy's `minimize` takes a function to **minimise** — so we minimise the negative of utility.

In [ ]:
# Parameters
alpha = 0.4   # preference weight on good 1
p1    = 3.0   # price of good 1
p2    = 5.0   # price of good 2
I     = 120.0 # income

# Negative utility (we minimise negative = maximise positive)
def neg_utility(x):
    x1, x2 = x
    if x1 <= 0 or x2 <= 0:
        return 1e10  # infeasible
    return -(x1**alpha * x2**(1 - alpha))

# Budget constraint: p1*x1 + p2*x2 = I  →  p1*x1 + p2*x2 - I = 0
budget_constraint = {"type": "eq", "fun": lambda x: p1 * x[0] + p2 * x[1] - I}

# Non-negativity bounds
bounds = [(0.01, None), (0.01, None)]

result = optimize.minimize(
    neg_utility,
    x0=[10.0, 10.0],       # initial guess
    method="SLSQP",
    bounds=bounds,
    constraints=budget_constraint,
)

x1_star, x2_star = result.x
u_star = x1_star**alpha * x2_star**(1 - alpha)

print("Utility Maximisation Result:")
print(f"  Optimal x1* = {x1_star:.4f}")
print(f"  Optimal x2* = {x2_star:.4f}")
print(f"  Utility U*  = {u_star:.4f}")
print(f"  Budget check: {p1*x1_star + p2*x2_star:.2f} ≈ {I}")

# Analytical solution for comparison: x1* = alpha*I/p1, x2* = (1-alpha)*I/p2
x1_analytic = alpha * I / p1
x2_analytic = (1 - alpha) * I / p2
print(f"\nAnalytical solution: x1* = {x1_analytic:.4f}, x2* = {x2_analytic:.4f}")

---
## 2. Equilibrium Finding — Supply and Demand

`optimize.brentq` finds the root of a function (where it equals zero). Market equilibrium is where excess demand = 0.

In [ ]:
def demand(price):
    """Quantity demanded as a non-linear function of price."""
    return 100 / (1 + 0.5 * price) + 10

def supply(price):
    """Quantity supplied."""
    return 5 * price + 2

def excess_demand(price):
    return demand(price) - supply(price)

# Find equilibrium price where excess demand = 0
p_star = optimize.brentq(excess_demand, a=0.1, b=50)
q_star = demand(p_star)

print(f"Equilibrium price:    P* = {p_star:.4f}")
print(f"Equilibrium quantity: Q* = {q_star:.4f}")
print(f"Excess demand at P*:  {excess_demand(p_star):.2e}  (should be ~0)")

# Plot supply and demand curves
prices = np.linspace(0.1, 20, 200)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(demand(prices), prices, color="blue", linewidth=2, label="Demand")
ax.plot(supply(prices), prices, color="red",  linewidth=2, label="Supply")
ax.scatter([q_star], [p_star], s=100, zorder=5, color="black", label=f"Equilibrium (P={p_star:.2f}, Q={q_star:.2f})")
ax.set_xlabel("Quantity"); ax.set_ylabel("Price")
ax.set_title("Market Equilibrium")
ax.legend()
plt.tight_layout(); plt.show()

---
## 3. Numerical Integration — Consumer Surplus

In [ ]:
# Consumer surplus = integral of demand from p_star to p_max
# We integrate over price (holding the demand curve as quantity demanded at each price)
def consumer_surplus(p_star, p_max=20):
    """Area under demand curve above equilibrium price."""
    integrand = lambda p: demand(p)  # quantity at each price
    cs, error = integrate.quad(integrand, p_star, p_max)
    return cs, error

cs, err = consumer_surplus(p_star)
print(f"Consumer surplus at equilibrium price {p_star:.2f}: {cs:.2f} (integration error: {err:.2e})")

# How does CS change when price rises by 1?
cs_new, _ = consumer_surplus(p_star + 1)
print(f"Consumer surplus after price rise to {p_star+1:.2f}: {cs_new:.2f}")
print(f"Deadweight loss from $1 price increase: {cs - cs_new:.2f}")

---
## 4. ODEs — The Solow Growth Model

The Solow model says capital per worker evolves as:

`dk/dt = s * f(k) - (δ + n + g) * k`

where `f(k) = k^alpha`, `s` is the savings rate, `δ` is depreciation, `n` is population growth, `g` is technology growth.

In [ ]:
from scipy.integrate import odeint

# Solow parameters
alpha = 0.33   # capital share
s     = 0.25   # savings rate
delta = 0.05   # depreciation
n     = 0.01   # population growth
g     = 0.02   # technology growth

def solow(k, t):
    """RHS of Solow ODE: dk/dt"""
    return s * k**alpha - (delta + n + g) * k

# Initial conditions and time grid
k0   = 1.0   # low initial capital per effective worker
t    = np.linspace(0, 200, 500)

k_path = odeint(solow, k0, t).flatten()

# Steady-state capital: s*k^alpha = (delta+n+g)*k  →  k* = (s/(delta+n+g))^(1/(1-alpha))
k_star = (s / (delta + n + g))**(1 / (1 - alpha))
y_path = k_path**alpha  # output per effective worker

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t, k_path, linewidth=2, color="steelblue")
axes[0].axhline(k_star, linestyle="--", color="red", label=f"k* = {k_star:.2f}")
axes[0].set_xlabel("Time"); axes[0].set_ylabel("Capital per eff. worker")
axes[0].set_title("Solow Model: Capital Convergence")
axes[0].legend()

axes[1].plot(t, y_path, linewidth=2, color="green")
axes[1].axhline(k_star**alpha, linestyle="--", color="red", label=f"y* = {k_star**alpha:.2f}")
axes[1].set_xlabel("Time"); axes[1].set_ylabel("Output per eff. worker")
axes[1].set_title("Solow Model: Output Convergence")
axes[1].legend()

plt.tight_layout()
plt.savefig("solow_model.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"Analytical steady state k* = {k_star:.4f}")

---
## 5. Statistical Tests — Comparing Growth Rates

In [ ]:
np.random.seed(99)

# Two groups of countries: high-savings vs low-savings
high_savings_growth = np.random.normal(loc=3.5, scale=1.2, size=30)
low_savings_growth  = np.random.normal(loc=2.0, scale=1.4, size=30)

# Two-sample t-test: H0 = same mean growth
t_stat, p_value = stats.ttest_ind(high_savings_growth, low_savings_growth)

print("Two-sample t-test: High-savings vs Low-savings growth rates")
print(f"  High-savings mean: {high_savings_growth.mean():.2f}%")
print(f"  Low-savings mean:  {low_savings_growth.mean():.2f}%")
print(f"  t-statistic: {t_stat:.3f}")
print(f"  p-value:     {p_value:.4f}")
print(f"  Conclusion:  {'Reject H0' if p_value < 0.05 else 'Fail to reject H0'} at 5% significance")

# Normality test
stat_n, p_n = stats.shapiro(high_savings_growth)
print(f"\nShapiro-Wilk normality test (high-savings): p = {p_n:.3f}")
print(f"  Distribution appears {'normal' if p_n > 0.05 else 'non-normal'}")

---
## Your Turn — Profit Maximisation

A monopolist faces demand `Q = 200 - 2P` and has total cost `TC = Q^2 / 4 + 20*Q + 100`.

1. Express profit as a function of quantity `Q`
2. Use `scipy.optimize.minimize_scalar` to find the profit-maximising `Q*`
3. Compute the monopoly price `P*` and profit `π*`
4. Compare to perfect competition (where P = MC)

In [ ]:
# Your code here


In [ ]:
#@title Solution
def price_from_quantity(Q):
    """Inverse demand: P = (200 - Q) / 2"""
    return (200 - Q) / 2

def total_cost(Q):
    return Q**2 / 4 + 20 * Q + 100

def neg_profit(Q):
    revenue = price_from_quantity(Q) * Q
    return -(revenue - total_cost(Q))

result = optimize.minimize_scalar(neg_profit, bounds=(0, 200), method="bounded")
Q_star = result.x
P_star = price_from_quantity(Q_star)
profit_star = -result.fun

print("Monopoly Solution:")
print(f"  Q* = {Q_star:.2f}")
print(f"  P* = {P_star:.2f}")
print(f"  π* = {profit_star:.2f}")

# Perfect competition: P = MC = Q/2 + 20, with P = (200-Q)/2
# Solve: (200-Q)/2 = Q/2 + 20 → 100 - Q/2 = Q/2 + 20 → Q_pc = 80
Q_pc = 80
P_pc = price_from_quantity(Q_pc)
profit_pc = P_pc * Q_pc - total_cost(Q_pc)
print(f"\nPerfect Competition: Q={Q_pc}, P={P_pc:.2f}, π={profit_pc:.2f}")
print(f"Monopoly markup: P_monopoly/P_competitive = {P_star/P_pc:.2f}x")

---
## Summary

| Problem | SciPy function |
|---|---|
| Constrained optimisation (utility max) | `scipy.optimize.minimize` with constraints |
| Root finding (market clearing) | `scipy.optimize.brentq` |
| Unconstrained scalar optimisation | `scipy.optimize.minimize_scalar` |
| Numerical integration (consumer surplus) | `scipy.integrate.quad` |
| Differential equations (Solow) | `scipy.integrate.odeint` |
| Statistical tests | `scipy.stats.ttest_ind`, `.shapiro` |

**Next:** NB_06_StatsModels.ipynb — OLS, IV, panel data, and time-series econometrics.